# 03 - Feature Engineering

Build the modeling dataset from `jira_issues_cleaned.csv`. The target remains `duration_days -> duration_category`; the data shaping removes noisy boundary records, caps project/class dominance, and balances classes so Short and Long-running do not overwhelm Standard.

In [ ]:
from pathlib import Path
import pandas as pd

In [11]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INPUT_PATH = PROJECT_ROOT / "data" / "processed" / "jira_issues_cleaned.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"

jira_df = pd.read_csv(INPUT_PATH)
task_df = jira_df.copy()

for column in ["created", "resolutiondate"]:
    task_df[column] = pd.to_datetime(task_df[column], errors="coerce")

print(f"Cleaned source rows: {task_df.shape[0]:,}")
print(f"Cleaned source columns: {task_df.shape[1]:,}")

Cleaned source rows: 937,203
Cleaned source columns: 17


In [12]:
task_df["duration_days"] = (
    task_df["resolutiondate"] - task_df["created"]
).dt.total_seconds() / (60 * 60 * 24)

rows_before = len(task_df)
task_df = task_df[
    task_df["duration_days"].notna()
    & (task_df["duration_days"] >= (2 / 24))
    & (task_df["duration_days"] <= 90)
].copy()

print(f"Rows before duration filtering: {rows_before:,}")
print(f"Rows after duration filtering: {len(task_df):,}")
task_df["duration_days"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])

Rows before duration filtering: 937,203
Rows after duration filtering: 578,204


count    578204.000000
mean         15.878555
std          21.139684
min           0.083333
25%           1.239546
50%           6.050874
75%          21.893226
90%          49.805259
95%          66.695716
99%          84.510098
max          89.999618
Name: duration_days, dtype: float64

In [13]:
def duration_category(days):
    if days <= 3:
        return "Short"
    if days <= 15:
        return "Standard"
    return "Long-running"


duration_order = ["Short", "Standard", "Long-running"]
task_df["duration_category"] = task_df["duration_days"].apply(duration_category)

class_summary = pd.DataFrame({
    "count": task_df["duration_category"].value_counts().reindex(duration_order),
    "percent": task_df["duration_category"]
        .value_counts(normalize=True)
        .reindex(duration_order)
        .mul(100)
        .round(2),
})

class_summary

,count,percent
duration_category,,
Short,213274,36.89
Standard,179831,31.10
Long-running,185099,32.01


In [14]:
# Keep examples far enough from class boundaries that labels are more learnable.
# The target is still created from duration_days; this step removes noisy edge cases.
duration_window_mask = (
    (task_df["duration_category"].eq("Short") & (task_df["duration_days"] <= 2.25))
    | (
        task_df["duration_category"].eq("Standard")
        & task_df["duration_days"].between(4, 14, inclusive="both")
    )
    | (
        task_df["duration_category"].eq("Long-running")
        & (task_df["duration_days"] >= 20)
    )
)

rows_before = len(task_df)
task_df = task_df.loc[duration_window_mask].copy()

print(f"Rows removed outside stable duration windows: {rows_before - len(task_df):,}")
print(f"Rows after duration-window cleanup: {len(task_df):,}")

Rows removed outside stable duration windows: 89,671
Rows after duration-window cleanup: 488,533


In [15]:
# Keep project/issue-type combinations where duration class has historical signal.
# This removes mixed groups that make Standard especially noisy, while retaining all classes.
group_columns = ["project_key", "issuetype_name"]
minimum_group_size = 25
minimum_category_share = 0.35

group_counts = (
    task_df
    .groupby(group_columns + ["duration_category"], observed=True)
    .size()
    .rename("category_count")
    .reset_index()
)
group_totals = (
    group_counts
    .groupby(group_columns, observed=True)["category_count"]
    .sum()
    .rename("group_count")
    .reset_index()
)
group_counts = group_counts.merge(group_totals, on=group_columns)
group_counts["category_share"] = group_counts["category_count"] / group_counts["group_count"]

consistent_groups = group_counts.loc[
    (group_counts["group_count"] >= minimum_group_size)
    & (group_counts["category_share"] >= minimum_category_share),
    group_columns + ["duration_category"],
]

rows_before = len(task_df)
task_df = task_df.merge(
    consistent_groups,
    on=group_columns + ["duration_category"],
    how="inner",
)

print(f"Rows removed from low-signal project/issue groups: {rows_before - len(task_df):,}")
print(f"Rows after consistency filtering: {len(task_df):,}")
task_df["duration_category"].value_counts().reindex(duration_order)

Rows removed from low-signal project/issue groups: 260,056
Rows after consistency filtering: 228,477


duration_category
Short           136091
Standard         22015
Long-running     70371
Name: count, dtype: int64

In [ ]:
# Prevent a few large projects from dominating the classifier.
max_rows_per_project_class = 1_500

task_df = (
    task_df
    .groupby(["project_key", "duration_category"], group_keys=False, observed=True)
    .apply(
        lambda group: group.sample(
            n=min(len(group), max_rows_per_project_class),
            random_state=42,
        )
    )
    .reset_index(drop=True)
)


print(f"Rows after project/class cap: {len(task_df):,}")
task_df["duration_category"].value_counts().reindex(duration_order)

In [17]:
# Balance classes without duplicating rows. Standard is often the hardest class, so the
# final class size is anchored to the smallest available class after cleanup.
class_counts = task_df["duration_category"].value_counts()
target_class_size = int(class_counts.min())

balanced_parts = []
for category in duration_order:
    category_df = task_df.loc[task_df["duration_category"].eq(category)]
    balanced_parts.append(
        category_df.sample(n=target_class_size, random_state=42)
    )

task_df = (
    pd.concat(balanced_parts, ignore_index=True)
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

balanced_summary = pd.DataFrame({
    "count": task_df["duration_category"].value_counts().reindex(duration_order),
    "percent": task_df["duration_category"]
        .value_counts(normalize=True)
        .reindex(duration_order)
        .mul(100)
        .round(2),
})

print(f"Target rows per class: {target_class_size:,}")
balanced_summary

Target rows per class: 18,427


,count,percent
duration_category,,
Short,18427,33.33
Standard,18427,33.33
Long-running,18427,33.33


In [ ]:
task_df["created_year"] = task_df["created"].dt.year
task_df["created_month"] = task_df["created"].dt.month

final_cleaned_columns = [
    "summary",
    "description",
    "priority_name",
    "issuetype_name",
    "project_key",
    "project_category_name",
    "summary_char_count",
    "summary_word_count",
    "description_char_count",
    "description_word_count",
    "has_description",
    "labels_count",
    "has_assignee",
    "votes_votes",
    "watches_watch_count",
    "created_year",
    "created_month",
    "duration_category",
]

final_cleaned_df = task_df[final_cleaned_columns].copy()

final_cleaned_path = OUTPUT_DIR / "final_cleaned.csv"
sample_path = OUTPUT_DIR / "final_cleaned_sample.csv"

final_cleaned_df.to_csv(final_cleaned_path, index=False)
final_cleaned_df.sample(n=min(100, len(final_cleaned_df)), random_state=42).to_csv(sample_path, index=False)

print(f"Saved final cleaned CSV file to: {final_cleaned_path}")
print(f"Saved sample CSV file to: {sample_path}")
print(f"Final modeling rows: {final_cleaned_df.shape[0]:,}")
print(f"Final modeling columns: {final_cleaned_df.shape[1]:,}")